# Capítulo 9 — Saber quién habla

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Alguien escribe con el `cliente_id` de Laura y pregunta *«¿cómo preferís contactarme?»*. El agente le cuenta sus preferencias.

| | |
|---|---|
| **🧠 Componente agéntico** | Identidad y delegación |
| **La necesidad** | Que la identidad del cliente esté **verificada**: que no la escriba nadie en el payload ni la deduzca el modelo |
| **Lo que construimos** | JWT *inbound* en el Runtime (Cognito), `actor_id` desde el token y patrón OBO |
| **🏠 En local** | ☁️ No aplica: en tu portátil no hay clientes que identificar. |
| **☁️ Por qué AWS al salir a Internet** | En local, los secretos viven en un `.env` y los tokens se gestionan a mano. AgentCore Identity valida tokens, guarda credenciales en un Token Vault cifrado y permite actuar en nombre del usuario (OBO) sin superar sus permisos. |
| **En el ordenador sería…** | Las **cuentas de usuario** y el inicio de sesión del sistema operativo |
| **Requisitos del caso** | RNF-01, RNF-02 |
| **Prerrequisito · duración** | Capítulo 8 · 60 min |
| **⚠️ Dónde se rompe la analogía** | Iniciar sesión en un ordenador identifica a la persona; aquí hay **dos identidades**, la del agente y la del cliente, y confundirlas abre la puerta a los accesos indebidos. |

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(9)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(9)
# ponerse_al_dia(9)   # ← descomenta si llegas tarde o algo se rompió

## 🧠 La teoría: identidad y delegación

- En un sistema agéntico hay **dos identidades**: la del **agente** (qué software llama) y la del **usuario** (en nombre de quién).
- **Autenticación** (quién eres) frente a **autorización** (qué puedes hacer).
- **Delegación (OBO):** el agente actúa con los permisos del usuario, **nunca con más**.
- **Regla de oro:** la identidad no la escribe ni el modelo ni el usuario en un campo libre; la aporta la plataforma.

> Este componente **no existe en local**: en tu portátil no hay clientes que identificar.

## 1. Pool de clientes (Cognito) y usuarios de prueba

In [ ]:
import boto3, secrets

idp = boto3.client("cognito-idp", region_name=region())
if state_get("CLIENTES_POOL_ID"):
    print("El pool de clientes ya existe")
else:
    pool = idp.create_user_pool(PoolName="nova-clientes")["UserPool"]["Id"]
    client_id = idp.create_user_pool_client(
        UserPoolId=pool, ClientName="nova-web",
        ExplicitAuthFlows=["ALLOW_USER_PASSWORD_AUTH", "ALLOW_REFRESH_TOKEN_AUTH"])["UserPoolClient"]["ClientId"]
    password = "Nova-" + secrets.token_urlsafe(8) + "1a!"
    for user in ("C-001", "C-002"):
        idp.admin_create_user(UserPoolId=pool, Username=user, MessageAction="SUPPRESS")
        idp.admin_set_user_password(UserPoolId=pool, Username=user, Password=password, Permanent=True)
    state_set(CLIENTES_POOL_ID=pool, CLIENTES_CLIENT_ID=client_id, CLIENTES_PASSWORD=password,
              CLIENTES_DISCOVERY_URL=f"https://cognito-idp.{region()}.amazonaws.com/{pool}/.well-known/openid-configuration")

In [ ]:
import base64, json

tok = token_cliente("C-001")
claims = json.loads(base64.urlsafe_b64decode(tok.split(".")[1] + "=" * (-len(tok.split(".")[1]) % 4)))
print({k: claims[k] for k in ("username", "client_id", "token_use", "exp") if k in claims})

## 2. El agente obtiene la identidad del token

In [ ]:
%%writefile NovaAssist/app/NovaAgent/identidad.py
"""Identidad del cliente a partir del JWT que el Runtime YA ha validado (firma, emisor, caducidad)."""
import base64
import json


def claims_del_token(auth_header: str) -> dict:
    token = auth_header.split(" ", 1)[1]
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)
    return json.loads(base64.urlsafe_b64decode(payload))


def actor_desde_contexto(context) -> str | None:
    """Devuelve el username del token o None. Nunca usa datos del payload, que el cliente puede falsificar."""
    headers = getattr(context, "request_headers", None) or {}
    auth = headers.get("Authorization") or headers.get("authorization")
    if not auth or not auth.lower().startswith("bearer "):
        return None
    return claims_del_token(auth).get("username")

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.agent.conversation_manager import SlidingWindowConversationManager
from strands.models import BedrockModel

from contexto import contexto_sesion
from gateway import cliente_gateway, token_gateway
from identidad import actor_desde_contexto
from memoria import crear_session_manager, retrieval_config
from prompts import SYSTEM_PROMPT
from settings import MODEL_ID, REGION

app = BedrockAgentCoreApp()


def metricas(result):
    """Herramientas usadas y consumo de tokens (para la verificación en los notebooks)."""
    try:
        m = result.metrics
        return sorted(m.tool_metrics.keys()), dict(m.accumulated_usage)
    except Exception:  # noqa: BLE001
        return [], {}


@app.entrypoint
def invoke(payload, context):
    actor_id = actor_desde_contexto(context)  # identidad verificada; payload["cliente_id"] se IGNORA
    if not actor_id:
        return {"respuesta": "No se ha podido identificar al cliente.", "herramientas": [], "tokens": {}}
    prompt_sistema = SYSTEM_PROMPT + "\n\n" + contexto_sesion(payload.get("canal", "web"), payload.get("idioma", "es"))
    gateway = cliente_gateway(token_gateway())
    with gateway, crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.2),
            system_prompt=prompt_sistema,
            tools=gateway.list_tools_sync(),
            session_manager=sm,
            conversation_manager=SlidingWindowConversationManager(window_size=20),
            callback_handler=None,
        )
        result = agent(payload.get("prompt", ""))
    herramientas, uso = metricas(result)
    return {"respuesta": str(result), "herramientas": herramientas, "tokens": uso}


if __name__ == "__main__":
    app.run()

Prueba local: el token es de **C-001**, pero el payload dice **C-002**. El agente debe ignorar el payload.

In [ ]:
os.environ["NOVA_GATEWAY_TOKEN"] = token_m2m()
probar_local("¿Qué pedidos tengo y cómo preferís contactarme?", cliente_id="C-002",
             headers={"Authorization": f"Bearer {token_cliente('C-001')}"})
probar_local("Hola", headers={})   # sin token → no identificado
del os.environ["NOVA_GATEWAY_TOKEN"]

## 3. Autorizador JWT del Runtime y cabecera `Authorization`

Configura en el Runtime:
1. **Autorizador JWT** con el *discovery URL* y el *client id* del pool de clientes (valores en la celda siguiente).
2. **Allowlist de la cabecera `Authorization`**, para que el código reciba el token validado.

Hay dos vías:
- **Declararlo en `agentcore.json`** (recomendado). La entrada del runtime admite configuración de autorizador análoga a la del Gateway; consulta el esquema y valida con `agentcore validate`.
- **Consola de AgentCore.** Ojo: un `agentcore deploy` posterior puede revertir lo que cambies a mano.

In [ ]:
print("discoveryUrl :", state_get("CLIENTES_DISCOVERY_URL"))
print("allowedClients:", [state_get("CLIENTES_CLIENT_ID")])
print(json.dumps(agentcore_json().get("runtimes", []), indent=2, ensure_ascii=False))

In [ ]:
!cd "{PROJECT}" && agentcore validate && agentcore deploy --yes

## 4. Verificación

In [ ]:
try:
    invocar("Hola")   # SigV4 sin token: con autorizador JWT debe rechazarse
except Exception as e:
    print("Sin token →", type(e).__name__, e)

TOK_C1, TOK_C2 = token_cliente("C-001"), token_cliente("C-002")
invocar("¿Cómo preferís contactarme si hay un problema?", token=TOK_C1, cliente_id="C-002")  # usa la memoria de C-001
invocar("¿Cómo preferís contactarme si hay un problema?", token=TOK_C2)                      # no conoce las de C-001

## 5. Cerrar el acceso indebido (RNF-01): el patrón correcto

El Gateway recibe el token **M2M del agente**: sabe *qué agente* llama, pero no *en nombre de qué cliente*. Para que el sistema de pedidos filtre por propietario:

- **Producción:** usa ***token exchange* On-Behalf-Of (OBO)** de AgentCore Identity, que obtiene para el Gateway un token con la identidad del cliente. Así Policy y la Lambda ven al cliente real.
- **Mínimo:** propaga la identidad por un canal que el modelo no controle y comprueba en la Lambda `pedido.cliente_id == cliente_autenticado`.

**Regla de oro:** nunca confíes en un identificador que el modelo pueda escribir como argumento de una herramienta.

**Ejercicio:** repite la prueba del capítulo 2 (`TN-1002` con el token de C-001) y explica por qué sigue funcionando hasta que implementes OBO.

## Para pensar

- ¿Qué diferencia hay entre autenticar al agente (M2M) y autorizar al usuario (OBO)?
- ¿Por qué decodificar el JWT sin verificar la firma solo es aceptable aquí?

## 🔎 Lo que acabamos de descubrir

Ya sabemos quién es Laura… y aun así **nada impide** que el agente le reembolse 450 € (D2), ni que obedezca a la «autorización» que alguien dejó en la memoria (D5) o a un documento envenenado (D8).

In [ ]:
deuda("D4", "pagada", 9, "actor_id desde el JWT validado")
deuda("D7", "pagada", 9, "autorizador JWT en el Runtime")
deuda("D1", "parcial", 9, "identidad verificada; falta propagarla al Gateway con OBO")
cuaderno()
taximetro_cerrar(9)
guardar_partida(9)

---
➡️ **Siguiente capítulo:** `10_policy_guardrails.ipynb` — *Reglas que el modelo no puede saltarse*